# RNN&LSTM: 



- The 'easy' way (torch api): 
    - using `nn.RNN` and `nn.LSTM` directly. 
    - Goal: Understanding inputs/outputs shapes.
- The 'advanced' way (manual layer): 
    - re-implementing the math of RNNs/LSTMs using simple `nn.Linear` layers. 
    - Goal: Understanding the math
- Learning rate schedulers and the training pipeline
    - Controlling how the model learns over time using `StepLR`
    - Goal:
        - get familiar with the whole pipeline
        - understand what a scheduler can do

# RNN and LSTM

In [ ]:
from turtle import forward
import torch 
import torch.nn as nn

class RNN(nn.Module):

    def __init__(self, input_size=10, hidden_size=20, num_layers=1, num_classes=15):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.act = nn.Sigmoid()
        self.proj = nn.Linear(in_features=hidden_size, out_features=num_classes)

    def forward(self, x: torch.Tensor):
        # shape of x?
        # [3, 5, 10]
        out, hidden = self.rnn(x)
        # shape of out?
        # out, [3, 5, 20]
        act_out = self.act(out)
        # act_out [3, 5, 20]
        logits = self.proj(act_out)
        # logits [3, 5, 15]

        # hidden[1, 3, 20]
        return logits, hidden

In [ ]:
x = torch.randn((3, 5, 10))
# x [5, 3, 10]
print(x.shape)
rnn = RNN(input_size=10, hidden_size=20, num_layers=1, num_classes=15)
logits, hidden = rnn(x)
print(logits.shape, hidden.shape)

torch.Size([3, 5, 10])
torch.Size([3, 5, 15]) torch.Size([1, 3, 20])


In [ ]:
import torch
import torch.nn as nn

class LSTM(nn.Module):

    def __init__(self, input_size=10, hidden_size=20, num_layers=1, num_classes=15):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.act = nn.Sigmoid()
        self.proj = nn.Linear(in_features=hidden_size, out_features=num_classes)

    def forward(self, x: torch.Tensor):
        # shape of x?
        # [3, 5, 10]
        out, hidden = self.rnn(x)
        # shape of out?
        # out, [3, 5, 20]
        act_out = self.act(out)
        # act_out [3, 5, 20]
        logits = self.proj(act_out)
        # logits [3, 5, 15]

        # hidden[1, 3, 20]
        return logits, hidden

In [ ]:
input = torch.randn((3, 5, 10))
# [5, 3, 10]
# rearrange_input
# 
lstm = LSTM()
logits, hidden = lstm(input)
print(logits.shape)

torch.Size([3, 5, 15])


In [ ]:
hidden_states = hidden[0]
cell_states = hidden[1]
print(hidden_states.shape)
print(cell_states.shape)

torch.Size([1, 3, 20])
torch.Size([1, 3, 20])


# RNN & LSTM (nn.Linear)

## RNN

In [ ]:
import torch
import torch.nn as nn

class RawRNN(nn.Module):

    def __init__(self, input_size=10, hidden_size=20, num_layers=1, num_classes=15):
        super().__init__()
        self.w_i = nn.Linear(in_features=input_size, out_features=hidden_size)
        self.w_h = nn.Linear(in_features=hidden_size, out_features=hidden_size)
        self.act = nn.Sigmoid()
        self.proj = nn.Linear(in_features=hidden_size, out_features=num_classes)
        self.hidden_size = hidden_size

    def forward(self, x: torch.Tensor):
        # hidden [num_layer, batch_size, hidden_size]
        # hidden [1, batch_size, hidden_size] --> [1, 3, 20]
        # x: [3, 5, 10]
        batch_size, seq_len, input_size = x.shape
        memory = torch.zeros([batch_size, self.hidden_size], device=x.device)

        outputs = []

        for t in range(seq_len):
            # t = 1 [0, 1]
            # x [batch, seq, input_size]
            x_t = x[:, t, :] # [3, 10]

            wixt = self.w_i(x_t)
            wht = self.w_h(memory)
            a_t = self.act(wixt + wht)

            y_t = self.proj(a_t)
            outputs.append(y_t)

            memory = a_t

        # outputs: list of tensors, each tensor [3, 15]
        # [3, 5, 15] 
        # [dim_0, dim_1, dim_2]
        output = torch.stack(outputs, dim=1)
        return output, memory


In [ ]:
batch_size = 3
seq_len = 5
input_size = 10
input = torch.randn((batch_size, seq_len, input_size))
rawrnn = RawRNN()
output, memory = rawrnn(input)

# output: [3, 5, 15]
print(output.shape)

torch.Size([3, 5, 15])


In [ ]:
print(memory.shape)

torch.Size([3, 20])


## LSTM

In [ ]:
import torch
import torch.nn as nn

class RawLSTM(nn.Module):

    def __init__(self, input_size=10, hidden_size=20, num_layers=1, num_classes=15):
        super().__init__()

        concat_size = hidden_size + input_size
        # forget gate
        self.w_f = nn.Linear(in_features=concat_size, out_features=hidden_size)
        self.f_act = nn.Sigmoid()

        # input gate
        self.w_i = nn.Linear(in_features=concat_size, out_features=hidden_size)
        self.i_act = nn.Sigmoid()

        # cell
        self.w_c = nn.Linear(in_features=concat_size, out_features=hidden_size)
        self.c_act = nn.Tanh()

        # output gate
        self.w_o = nn.Linear(in_features=concat_size, out_features=hidden_size)
        self.o_act = nn.Sigmoid()

        # final act
        self.final_act = nn.Tanh()

        # proj
        self.proj = nn.Linear(in_features=hidden_size, out_features=num_classes)

        self.hidden_size = hidden_size

    def forward(self, x: torch.Tensor):
        # x: [3, 5, 10]
        batch_size, seq_len, input_size = x.shape 
        # [batch_size, hidden_size]
        cell = torch.zeros((batch_size, self.hidden_size), device=x.device)
        hidden = torch.zeros((batch_size, self.hidden_size), device=x.device)

        outputs = []

        for t in range(seq_len):
            # x: [3, 5, 10]
            x_t = x[:, t, :] # [3, 10]

            # hidden: 3, 20
            # x_t:    3, 10
            concat = torch.cat([hidden, x_t], dim=1) # [3, 30]         
            
            # forget gate
            f_t = self.f_act(self.w_f(concat)) # [3, 20]
            # input gate
            i_t = self.i_act(self.w_i(concat)) # [3, 20]
            # output gate
            o_t = self.o_act(self.w_o(concat)) # [3, 20]

            # intermediate step
            c_inter = self.c_act(self.w_c(concat)) # [3, 20]
            # cell: [3, 20] c_inter: [3, 20]
            c_t = f_t * cell + i_t * c_inter # [3, 20]

            # output
            h_t = o_t * self.final_act(c_t) # [3, 20]

            cell = c_t
            hidden = h_t

            logits = self.proj(h_t) # [3, 15]
            outputs.append(logits)

        # [3, 5, 15]
        # [dim0, dim1, dim2]
        output = torch.stack(outputs, dim=1)

        return output, (hidden, cell)


        


In [ ]:
# [3, 5, 10]
input = torch.randn((3, 5, 10))
rawlstm = RawLSTM()
output, (hidden, cell) = rawlstm(input)

# [3, 5, 15]
print(output.shape)

torch.Size([3, 5, 15])


In [ ]:
print(hidden.shape) # [3, 20]
print(cell.shape)

torch.Size([3, 20])
torch.Size([3, 20])


# Learning Rate Scheduler

In [ ]:
# data prepara
batch_size = 3
seq_len = 5
input_size = 10
first_batch = torch.randn((batch_size, seq_len, input_size))
second_batch = torch.randn((batch_size, seq_len, input_size))
# 3, 5 
# 15: 0, 1, 2, ..., 14
first_labels = torch.tensor([
    [3, 5, 1, 1, 6],
    [2, 4, 7, 11, 3],
    [12, 13, 14, 0, 1],
], dtype=int)

second_labels = torch.tensor([
    [12, 7, 11, 11, 4],
    [9, 1, 5, 3, 5],
    [10, 9, 2, 5,8],
], dtype=int)

dataset = [[first_batch, first_labels], [second_batch, second_labels]]

In [ ]:
print(second_labels.shape)

torch.Size([3, 5])


In [ ]:
model = RawLSTM()
out, hidden = model(first_batch)
print(out.shape)

torch.Size([3, 5, 15])


In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
learning_rate = 0.001
adam = torch.optim.Adam(model.parameters(), learning_rate)

In [ ]:
# epoch 0: lr: 0.001 1e-3
# epoch 1: lr = old_lr * 0.1 = 0.0001 1e-4
# step LR

step_lr = torch.optim.lr_scheduler.StepLR(
    optimizer=adam, 
    step_size=1, 
    gamma=0.1
)

In [ ]:
num_epoch = 2

for epoch in range(num_epoch):
    for batch, labels in dataset:
        adam.zero_grad()
        out, hidden = model(batch)
        #       dim 0,     dim 1,     dim 2
        # out: batch_size, seq_len, num_classes -> batch_size, num_classes, seq_len
        # labels: batch_size, num_classes
        logits = out.permute(0, 2, 1)
        loss = loss_fn(logits, labels)
        loss.backward()
        adam.step()

        current_lr = adam.param_groups[0]['lr']
        print('epoch', epoch, 'learning rate', current_lr)
    step_lr.step()

epoch 0 learning rate 0.001
epoch 0 learning rate 0.001
epoch 1 learning rate 0.0001
epoch 1 learning rate 0.0001
